In [1]:
# ══════════════════════════════════════════════════════════════════
# 1. CARGA DE LIBRERÍAS Y DATOS
# ══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, kstest, norm
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

# Estilo global de gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

In [3]:
from pathlib import Path

# Localiza data/raw desde el directorio actual o alguno de sus directorios superiores.
base = Path.cwd()
rutas_candidatas = [
    ruta
    for directorio in (base, *base.parents)
    for ruta in (directorio / "data" / "raw", directorio / "raw")
]
carpeta = next((ruta for ruta in rutas_candidatas if ruta.is_dir()), None)

if carpeta is None:
    raise FileNotFoundError(f"No se encontró data/raw desde {Path.cwd()}")

# Cargar únicamente los años solicitados: 2022, 2023, 2024, 2025 y 2026.
anios = [2021, 2022, 2023, 2024, 2025, 2026]
datos = []

for anio in anios:
    carpeta_anio = carpeta / f"data{anio}"
    if not carpeta_anio.is_dir():
        print(f"Aviso: no existe la carpeta {carpeta_anio}; se omite.")
        continue

    archivos_anio = sorted(
        archivo for archivo in carpeta_anio.iterdir()
        if archivo.is_file() and archivo.suffix.lower() in {".csv", ".xlsx"}
    )

    for archivo in archivos_anio:
        try:
            if archivo.suffix.lower() == ".csv":
                df = pd.read_csv(archivo, sep=";")
            else:
                df = pd.read_excel(archivo)
        except Exception as error:
            raise ValueError(f"No se pudo leer el archivo: {archivo}") from error

        df["anio_archivo"] = anio
        datos.append(df)

if not datos:
    raise FileNotFoundError(
        f"No se encontraron archivos CSV o XLSX para los años {anios} en {carpeta}"
    )

raw = pd.concat(datos, ignore_index=True, sort=False)
raw.head()

,Fecha,provincia,Canton,Cod_Parroquia,Parroquia,Servicio,Subtipo,anio_archivo,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,7/8/2021,GUAYAS,GUAYAQUIL,90150.0,"GUAYAQUIL, CABECERA CANTONAL Y CAPITAL PROVINCIAL",Seguridad Ciudadana,Libadores,2021,NaN,NaN,NaN,NaN
1,7/8/2021,GUAYAS,GUAYAQUIL,90150.0,"GUAYAQUIL, CABECERA CANTONAL Y CAPITAL PROVINCIAL",Seguridad Ciudadana,Diversas formas de explotación,2021,NaN,NaN,NaN,NaN
2,7/8/2021,GUAYAS,GUAYAQUIL,90150.0,"GUAYAQUIL, CABECERA CANTONAL Y CAPITAL PROVINCIAL",Seguridad Ciudadana,Consumo de sustancias sujetas a fiscalización,2021,NaN,NaN,NaN,NaN
3,7/8/2021,GUAYAS,GUAYAQUIL,90150.0,"GUAYAQUIL, CABECERA CANTONAL Y CAPITAL PROVINCIAL",Seguridad Ciudadana,Patrullaje policial en el sector solicitado,2021,NaN,NaN,NaN,NaN
4,7/8/2021,GUAYAS,GUAYAQUIL,90150.0,"GUAYAQUIL, CABECERA CANTONAL Y CAPITAL PROVINCIAL",Seguridad Ciudadana,Diversas formas de explotación,2021,NaN,NaN,NaN,NaN
